# 書籍公開記念記事
## 地形縞による地形推定

記事リンク: []()

## 概要
単純な干渉解析処理による地形縞から DEMを作成する処理です。

## 使用データ


| 項目 | 情報 |
| ---- | ---- |
| 衛星 | ALOS PALSAR |
| 観測シーンID　| メイン:`ALPSRP216690620`, サブ:`ALPSRP223400620` |
| 日時 | メイン:`2010/02/17`, サブ:`2010/04/04` |
| データリンク | メイン:[ASF](https://search.asf.alaska.edu/#/?maxResults=250&dataset=SENTINEL-1&searchType=List%20Search&zoom=9.268&center=130.791,31.211&searchList=ALPSRP216690620&resultsLoaded=true&granule=ALPSRP216690620-L2.2), サブ:[ASF](https://search.asf.alaska.edu/#/?maxResults=250&dataset=SENTINEL-1&searchType=List%20Search&zoom=9.268&center=130.791,31.211&searchList=ALPSRP223400620&resultsLoaded=true&granule=ALPSRP223400620-L2.2) Level 1.1 Image |
| 画像クレジット| © JAXA |

In [ ]:
import os
import numpy as np
import warnings
import tifffile
import scipy
import cv2

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt

In [ ]:
PATH_ROOT_ALOS_M = '../data/sakurazima/ALPSRP216690620-H1.1__A/'
PATH_ROOT_ALOS_S = '../data/sakurazima/ALPSRP223400620-H1.1__A/'

PATH_OUTPUT = os.path.join('output', '1_5_999', 'dem')
os.makedirs(PATH_OUTPUT, exist_ok=True)

In [ ]:
from utils.ceos_io import ALOSPALSARData

alos_m = ALOSPALSARData(PATH_ROOT_ALOS_M)
slc_m = alos_m.read_slc()

alos_s = ALOSPALSARData(PATH_ROOT_ALOS_S)
slc_s = alos_s.read_slc()

In [ ]:
plt.figure(figsize=(16, 12), dpi=120, facecolor='w', edgecolor='k')
plt.subplot(1, 2, 1)
plt.title('ALOS PALSAR Intencity', fontsize=12, fontweight='bold')
plt.imshow(np.abs(slc_m[0]), cmap='gray')
plt.colorbar(shrink=0.6)
plt.subplot(1, 2, 2)
plt.title('ALOS PALSAR Phase', fontsize=12, fontweight='bold')
plt.imshow(np.angle(slc_m[0]), cmap='hsv', vmin=-np.pi, vmax=np.pi, interpolation='nearest')
plt.colorbar(shrink=0.6)
plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, 'alos_m.png'))
plt.show();
plt.clf()
plt.close()

In [ ]:
pos_m = alos_m.read_location()
pos_s = alos_s.read_location()
print('ALOS `M`ain Path 4 edgs points:  \n', np.array(pos_m).reshape(4, 2), )

In [6]:
alos_m.write_single_look_complex(
    PATH_GEOTIF_SLC=os.path.join(PATH_OUTPUT, 'ALOS_m.tif'), 
    imgs=slc_m)

In [ ]:
bbox = [
    130.60652,31.61737,  # top left
    130.72007,31.54760,  # bottom right
]

slc_m_crop = alos_m.crop_geotiff(
    bbox,
    PATH_OUTPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'ALOS_m_HH_crop.tif'), 
    PATH_INPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'ALOS_m_HH.tif'),
)
slc_m_crop.shape

In [ ]:
plt.figure(figsize=(16, 12), dpi=160, facecolor='w', edgecolor='k')
plt.title('ALOS PALSAR Crop Area Intencity', fontsize=12, fontweight='bold')
plt.imshow(np.abs(slc_m_crop), cmap='gray', )
plt.colorbar(shrink=0.8)
plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, 'alos_m_crop_intencity.png'))
plt.show();plt.clf();plt.close()

## Orbit

In [ ]:
(orb_m, obs_m) = alos_m.read_orbit()
(orb_s, obs_s) = alos_s.read_orbit()

plt.figure(figsize=(10, 4), dpi=80, facecolor='w', edgecolor='k')
plt.plot(obs_m, orb_m[:, 0], label='Orbit X')
plt.plot(obs_m, orb_m[:, 1], label='Orbit Y')
plt.plot(obs_m, orb_m[:, 2], label='Orbit Z')
plt.title('ALOS PALSAR ECEF Orbit', fontsize=12, fontweight='bold')
plt.xlabel('Milli Sec (Time)')
plt.ylabel('Height (m)')
plt.legend()
plt.grid()
plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, 'alos_m_orbit.png'))
plt.show();plt.clf();plt.close()

for idx_axis, axis in enumerate(['X', 'Y', 'Z']):
    plt.figure(figsize=(8, 3), dpi=80, facecolor='w', edgecolor='k')
    plt.plot(obs_m, orb_m[:, idx_axis], label=f'Orbit {axis}')
    plt.title(f'ALOS PALSAR ECEF Orbit {axis}', fontsize=12, fontweight='bold')
    plt.xlabel('Milli Sec (Time)')
    plt.ylabel('Height (m)')
    plt.grid()
    plt.tight_layout(pad=1.3)
    plt.savefig(os.path.join(PATH_OUTPUT, f'alos_m_orbit_{axis}.png'))
    plt.show();plt.clf();plt.close()

In [ ]:
(A, B, CS, CL) = alos_m.read_geo_matrix()
print('ALOS `M`ain Path Geo Matrix:  \n', A.shape, B.shape, CS, CL)


In [ ]:
xyz_m_loc, (idx_sample, idx_line) = alos_m.read_xyz_raster(A, B, CS, CL)
print('ALOS `M`ain Path Geo Obsebasion Area Raster:  \n', xyz_m_loc.shape)

In [12]:
from utils.interferogram import get_slant_range

slant_m = get_slant_range(orb_m, xyz_m_loc, idx_line)

In [ ]:
plt.figure(figsize=(7, 10), dpi=70, facecolor='w', edgecolor='k')
plt.title('ALOS PALSAR Slant Range', fontsize=12, fontweight='bold')
plt.imshow(slant_m, cmap='autumn',)
plt.colorbar(shrink=0.8)
plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, 'alos_m_slant_range.png'))
plt.show();
plt.clf()
plt.close()

## Sub Path

In [14]:
from utils.interferogram import (
    interferogram, 
    orbital_stripe, 
    wrap_phase, 
    coregistoration_homomorphpy_cfloat, 
    to8bit,
    coregistration_phase_only_correlation
    )

In [ ]:
alos_s.write_single_look_complex(
    PATH_GEOTIF_SLC=os.path.join(PATH_OUTPUT, 'ALOS_s.tif'), 
    imgs=slc_s)
slc_s_crop = alos_s.crop_geotiff(
    bbox,
    PATH_OUTPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'ALOS_s_HH_crop.tif'), 
    PATH_INPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'ALOS_s_HH.tif'),
    height=slc_m_crop.shape[0],width=slc_m_crop.shape[1],
)
(orb_s, obs_s) = alos_s.read_orbit()
slant_s = get_slant_range(orb_s, xyz_m_loc, idx_line)
slant_m = get_slant_range(orb_m, xyz_m_loc, idx_line)
slant_s.shape, obs_m.shape, xyz_m_loc.shape, idx_line.shape

In [ ]:
RANGE_LEFT_PAD = 1
RANGE_RIGHT_PAD = 0
AZIMUTH_TOP_PAD = 0
AZIMUTH_BOTTOM_PAD = 0
RANGE_LEFT_CUT = 0
RANGE_RIGHT_CUT = 0
AZIMUTH_TOP_CUT = 0
AZIMUTH_BOTTOM_CUT = 0

shift_txt = f'P-R{RANGE_LEFT_PAD}-{RANGE_RIGHT_PAD}A{AZIMUTH_TOP_PAD}-{AZIMUTH_BOTTOM_PAD}'
shift_txt += f'_C-R{RANGE_LEFT_CUT}-{RANGE_RIGHT_CUT}A{AZIMUTH_TOP_CUT}-{AZIMUTH_BOTTOM_CUT}'
print(shift_txt)

H_S, W_S = slc_s[0].shape

# shift flow
slc_s_shift = np.pad(
    slc_s[0][
    AZIMUTH_TOP_CUT:H_S-AZIMUTH_BOTTOM_CUT,
    RANGE_LEFT_CUT:W_S-RANGE_RIGHT_CUT
    ],
    (
        (AZIMUTH_TOP_PAD, AZIMUTH_BOTTOM_PAD), 
        (RANGE_LEFT_PAD, RANGE_RIGHT_PAD)
        ),
    'constant',
    constant_values=0
    )

print(slc_s_shift.shape)

# recrop flow
alos_s.write_single_look_complex(
    PATH_GEOTIF_SLC=os.path.join(PATH_OUTPUT, 'ALOS_s_shift.tif'), 
    imgs=slc_s_shift,
    )

slc_s_shift_crop = alos_s.crop_geotiff(
    bbox,
    PATH_OUTPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'ALOS_s_shift_HH_crop.tif'), 
    PATH_INPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'ALOS_s_shift_HH.tif'),
    height=slc_m_crop.shape[0],width=slc_m_crop.shape[1],
)

In [ ]:
_, slc_s_crop_cor ,difference  = coregistration_phase_only_correlation(
        slc_m_crop, slc_s_shift_crop,
        s_left_pad=0, s_right_pad=0, s_top_pad=0, s_bottom_pad=0,
        
    )
# no shift -> (1, 1, 0, 0)

print('ALOS `S`ub Path Coregistration Difference:', difference)
print(f'Round Pixel : (H x W) = {round(difference[0])}, {round(difference[1])}')
print(f'Size of Main Path: {slc_m_crop.shape}')
print(f'Size of Sub  Path: {slc_s_crop.shape} -> {slc_s_crop_cor.shape}')

ifgm = interferogram(slc_m_crop, slc_s_crop_cor)

amp_s_crop_cor = to8bit(np.abs(slc_s_crop_cor))
amp_m_crop = to8bit(np.abs(slc_m_crop))
amp_ms_crop = np.stack([
    to8bit(np.abs(slc_m_crop)), amp_s_crop_cor, amp_s_crop_cor], 
                       axis=2)

plt.figure(figsize=(18, 8), dpi=160, facecolor='w', edgecolor='k')
plt.subplot(1, 2, 1)
plt.imshow(ifgm, cmap='jet',)
plt.colorbar(shrink=0.6)
plt.title(f'ALOS PALSAR Interferogram', fontsize=12, fontweight='bold')
plt.subplot(1, 2, 2)
plt.title(f'ALOS PALSAR Overlay Intencity', fontsize=12, fontweight='bold')
plt.imshow(amp_ms_crop)
plt.colorbar(shrink=0.6)
plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, f'ifgm_first_shift_{shift_txt}.png'))
plt.show();plt.clf();plt.close()

In [18]:
plt.imsave(os.path.join(PATH_OUTPUT, f'ifgm_shift_{shift_txt}.png'), ifgm, cmap='hsv')

In [ ]:
phase_rgb = cv2.imread(os.path.join(PATH_OUTPUT, f'ifgm_shift_{shift_txt}.png'))

plt.figure(figsize=(16, 10), dpi=160, facecolor='w', edgecolor='k')
plt.imshow(phase_rgb, alpha=0.9)
plt.imshow(amp_ms_crop, alpha=0.5)
plt.title(f'ALOS PALSAR Intencity & Interferogram', fontsize=12, fontweight='bold')
plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, f'ifgm_overlay_shift_{shift_txt}.png'))
plt.show();plt.clf();plt.close()

In [ ]:
plt.figure(figsize=(7, 10), dpi=70, facecolor='w', edgecolor='k')
plt.title('ALOS PALSAR Slant Range', fontsize=12, fontweight='bold')
plt.imshow(slant_s, cmap='autumn',)
plt.colorbar(shrink=0.8)
plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, 'alos_s_slant_range.png'))
plt.show();
plt.clf()
plt.close()

In [ ]:
wave_length = alos_m.wave_length
print('Wave Length λ: ', wave_length)

stripe_orb = orbital_stripe(slant_m, slant_s, wave_length)

plt.figure(figsize=(6, 8), dpi=80, facecolor='w', edgecolor='k')
plt.imshow(stripe_orb, cmap='gist_rainbow', vmin=-np.pi, vmax=np.pi, interpolation='nearest')
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Orbital Stripe', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'alos_orbital_stripe.png'))
plt.show();
plt.clf()
plt.close()

In [22]:
alos_s.write_geotiff(
    PATH_GEOTIF=os.path.join(PATH_OUTPUT, 'stripe_orbit.tif'), 
    img=stripe_orb,
)

In [23]:
# crop area
stripe_orb_crop = alos_s.crop_geotiff(
    bbox,
    PATH_OUTPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'stripe_orbit_crop.tif'), 
    PATH_INPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'stripe_orbit.tif'),
    height=slc_m_crop.shape[0],width=slc_m_crop.shape[1],
)

In [ ]:
plt.figure(figsize=(12, 8), dpi=70, facecolor='w', edgecolor='k')
plt.imshow(stripe_orb_crop, 
           cmap='jet', 
           vmin=-np.pi, vmax=np.pi, 
           )
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Orbital Stripe Crop Area', fontsize=12, fontweight='bold')
plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, 'orbital_stripe_crop.png'))
plt.show();plt.clf();plt.close()

In [ ]:
ifgm_rm_orb = wrap_phase(ifgm - stripe_orb_crop)

plt.figure(figsize=(12, 8), dpi=160, facecolor='w', edgecolor='k')
plt.imshow(ifgm_rm_orb, cmap='jet', vmin=-np.pi, vmax=np.pi)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Interferogram Remove Orbital Stripe', fontsize=12, fontweight='bold')
plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, f'ifgm_remove_orbital_stripe_shift_{shift_txt}.png'))
plt.show();plt.clf();plt.close()

In [26]:
plt.imsave(os.path.join(PATH_OUTPUT, 'ifgm-vis_remove_orbital_stripe_shift_{shift_txt}.png'), ifgm_rm_orb, cmap='jet')

In [ ]:
phase_rgb = cv2.imread(os.path.join(PATH_OUTPUT, 'ifgm-vis_remove_orbital_stripe_shift_{shift_txt}.png'))

plt.figure(figsize=(18,12), dpi=200, facecolor='w', edgecolor='k')
plt.title('Interferogrum Phase & Intency')
plt.imshow(phase_rgb, alpha=0.95)
plt.imshow(amp_ms_crop, alpha=0.75,)
plt.tight_layout()
plt.savefig(PATH_OUTPUT + f'/ifgm-overlay_rm_orbit_shift_{shift_txt}.png')
plt.show();plt.clf();plt.close()


## 地形差 - 地形縞   
``` 
130.60652,31.61737,  
130.72007,31.54760, 
```

In [ ]:
!gdalwarp -te 130.60652 31.54760 130.72007 31.61737 -t_srs EPSG:4326 -te_srs EPSG:4326 \
    ../data/sakurazima/srtm_dem_sakurazima_cut.tif ../data/sakurazima/srtm_dem_sakurazima_cut_height_area.tif

In [ ]:
PATH_DEM_CUT = '../data/sakurazima/srtm_dem_sakurazima_cut_height_area.tif'
dem = tifffile.imread(PATH_DEM_CUT).astype(np.float32)
dem = np.where(dem < 0, 0, dem) # preprocess
H_DEM, W_DEM = dem.shape

alos_m.crop_imgs = amp_ms_crop
dem = alos_m.resize_crop_area(dem)
H_DEM_RESIZE, W_DEM_RESIZE = dem.shape

W_SCALE = W_DEM_RESIZE / W_DEM
print('Longitude Scale of DEM: ', W_SCALE)

plt.figure(figsize=(14, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(dem, cmap='terrain', vmin=0, vmax=1117)
plt.colorbar(shrink=0.8)
plt.title('SRTM DEM 30m', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'dem.png'))
plt.show();plt.clf();plt.close()

In [30]:
xyz_elev_m_loc, (_, idx_line_elev) = alos_m.read_xyz_raster(
    A, B, CS, CL, dem=dem, 
    bbox=bbox, PATH_OUT=PATH_OUTPUT)

In [31]:
idx_line_elev_crop = alos_m.write_crop_geotif(
    PATH_GEOTIF=os.path.join(PATH_OUTPUT, 'ALOS_m_line-elev.tif'), 
    img=idx_line_elev.astype(np.float32), bbox=bbox)

idx_line_elev_crop = idx_line_elev_crop.astype(np.int32)

In [32]:
slant_elev_s_crop = get_slant_range(orb_s, xyz_elev_m_loc, idx_line_elev_crop)
slant_elev_m_crop = get_slant_range(orb_m, xyz_elev_m_loc, idx_line_elev_crop)

In [33]:
from osgeo import gdal

In [34]:
from utils.interferogram import slide_dem

dem_slide = slide_dem(slant_elev_m_crop, dem)

xyz_elev_slide_m_loc, (_, idx_line_elev_slide) = alos_m.read_xyz_raster(
    A, B, CS, CL, dem=dem_slide,
    bbox=bbox, PATH_OUT=PATH_OUTPUT
    )

idx_line_elev_slide_crop = alos_m.write_crop_geotif(
    PATH_GEOTIF=os.path.join(PATH_OUTPUT, 'ALOS_m_line-elev-slide.tif'), 
    img=idx_line_elev_slide.astype(np.float32), bbox=bbox)

idx_line_elev_slide_crop = idx_line_elev_slide_crop.astype(np.int32)

slant_elev_slide_s = get_slant_range(orb_s, xyz_elev_slide_m_loc, idx_line_elev_slide_crop)
slant_elev_slide_m = get_slant_range(orb_m, xyz_elev_slide_m_loc, idx_line_elev_slide_crop)

In [ ]:
plt.figure(figsize=(12, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(dem_slide, cmap='terrain', vmin=0, vmax=1117)
plt.colorbar(shrink=0.8)
plt.title('SRTM DEM 30m Slide', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'dem_slide.png'))
plt.show();plt.clf();plt.close()

In [36]:
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'dem_slide.tif'), dem_slide)

In [ ]:
plt.figure(figsize=(8, 6), dpi=70, facecolor='w', edgecolor='k')
plt.imshow(slant_elev_m_crop, cmap='spring',)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Slant Range from DEM', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'slant_elev_m.png'))
plt.show();plt.clf();plt.close()

In [38]:
alos_m.write_geotiff(
    PATH_GEOTIF=os.path.join(PATH_OUTPUT, 'slant_m.tif'), 
    img=slant_m,
    gdal_type=gdal.GDT_Float64
)
slant_m_cut = alos_m.crop_geotiff(
    bbox,
    PATH_OUTPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'slant_m_cut.tif'), 
    PATH_INPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'slant_m.tif'),
    height=slc_m_crop.shape[0],width=slc_m_crop.shape[1],
)
alos_m.write_geotiff(
    PATH_GEOTIF=os.path.join(PATH_OUTPUT, 'slant_s.tif'), 
    img=slant_s,
    gdal_type=gdal.GDT_Float64
)
slant_s_cut = alos_m.crop_geotiff(
    bbox,
    PATH_OUTPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'slant_s_cut.tif'), 
    PATH_INPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'slant_s.tif'),
    height=slc_m_crop.shape[0],width=slc_m_crop.shape[1],
)

In [ ]:
plt.figure(figsize=(10, 6), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(slant_m_cut - slant_elev_m_crop, cmap='gist_earth',)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Slant Range from DEM', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'slant_diff_m.png'))
plt.show();
plt.clf();plt.close()

In [40]:
from utils.interferogram import get_topography

topography = get_topography(
    slant_m_cut, slant_s_cut,
    slant_elev_slide_m, slant_elev_slide_s, 
    wave_length)

In [ ]:
plt.figure(figsize=(12, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(topography, cmap='hsv', vmin=-np.pi, vmax=np.pi)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Topography Phase', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'topography.png'))
plt.show();plt.clf();plt.close()

In [ ]:
ifgm_topo = wrap_phase(ifgm_rm_orb - topography)

plt.figure(figsize=(12, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(ifgm_topo, cmap='hsv', vmin=-np.pi, vmax=np.pi)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Interferogram removal Topography Phase', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'ifgm_topography.png'))
plt.show();plt.clf();plt.close()


In [43]:
plt.imsave(os.path.join(PATH_OUTPUT, 'ifgm-vis_topography.png'), ifgm_topo, cmap='hsv')

In [ ]:
phase_rgb_topo = cv2.imread(os.path.join(PATH_OUTPUT, 'ifgm-vis_topography.png'))

plt.figure(figsize=(18,12), dpi=200, facecolor='w', edgecolor='k')
plt.title('Interferogrum Phase & Intency')
plt.imshow(phase_rgb_topo, alpha=0.95)
# plt.colorbar(shrink=0.5, aspect=20)
plt.imshow(amp_ms_crop, alpha=0.75,)
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'ifgm-overlay_rm_topography.png'))
plt.show();plt.clf();plt.close()


In [45]:
from utils.interferogram import goldshtein_phase_filter_sliding_window
# from tqdm import tqdm

In [ ]:
clx_rm_topo = amp_m_crop * np.exp(1j * ifgm_topo)

slc_rm_topo_filter = goldshtein_phase_filter_sliding_window(clx_rm_topo, alpha=0.5, patch_size=32,)
ifgm_topo_filtered = np.angle(slc_rm_topo_filter)

plt.figure(figsize=(16, 10), dpi=200, facecolor='w', edgecolor='k')
plt.imshow(ifgm_topo_filtered, cmap='hsv', vmin=-np.pi, vmax=np.pi,)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Interferogram removal Topography Phase Filtered', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'ifgm_topography_filtered.png'))
plt.show();plt.clf();plt.close()

In [54]:
plt.imsave(os.path.join(PATH_OUTPUT, 'ifgm-vis_topography_filtered.png'), ifgm_topo_filtered, cmap='hsv')

In [ ]:
from utils.interferogram import coherence_multi_process

PATCH_SIZE = 2
coherence = coherence_multi_process(slc_m_crop, slc_s_crop_cor, PATCH_SIZE)

In [ ]:
plt.figure(figsize=(12, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(coherence, cmap='gnuplot', vmin=0, vmax=1)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Coherence', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'coherence.png'))
plt.show();
plt.clf();plt.close()

tifffile.imwrite(os.path.join(PATH_OUTPUT, 'coherence.tif'), coherence)

In [ ]:
th_coherence = 0.3

# plot histogram
plt.figure(figsize=(12, 4), facecolor='w', edgecolor='k')
plt.hist(coherence.ravel(), bins=200, color='b', range=(-0.05, 1.05), label='Coherence')
plt.title('Coherence Histogram', fontsize=12, fontweight='bold')
plt.grid()
plt.vlines(th_coherence, 0, 800000, 
           colors='red', linestyle='dashed', linewidth=3,
           label='Weak Threshold')
plt.tight_layout()
plt.legend()
plt.savefig(os.path.join(PATH_OUTPUT, 'coherence_hist.png'))
plt.show();plt.clf();plt.close()


In [ ]:
coherence_cut = np.where(coherence > th_coherence, coherence, 0)

# plot histogram
plt.figure(figsize=(12, 4), facecolor='w', edgecolor='k')
plt.hist(coherence_cut.ravel(), bins=200, color='b', range=(-0.05, 1.05), label='Coherence')
plt.title('Coherence Cut Histogram', fontsize=12, fontweight='bold')
plt.grid()
plt.vlines(th_coherence, 0, 800000, 
           colors='red', linestyle='dashed', linewidth=3,
           label='Weak Threshold')
plt.tight_layout()
plt.legend()
plt.savefig(os.path.join(PATH_OUTPUT, 'coherence_cut_hist.png'))
plt.show();plt.clf();plt.close()

In [ ]:
plt.figure(figsize=(12, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(coherence_cut, cmap='gnuplot', vmin=0, vmax=1)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Coherence Cut', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'coherence_cut.png'))
plt.show();plt.clf();plt.close()

In [60]:
import snaphu

In [ ]:
unw, conncomp = snaphu.unwrap(slc_rm_topo_filter, coherence,
                              nlooks=32.0, cost="smooth", init="mcf", nproc=8)

In [ ]:
num_plot_cycle = 10

plt.figure(figsize=(24, 12), dpi=120, facecolor='w', edgecolor='k')
plt.subplot(1, 2, 1)
plt.imshow(unw, cmap='coolwarm', vmin=-np.pi*num_plot_cycle, vmax=np.pi*num_plot_cycle)
plt.colorbar(shrink=0.5)
plt.title('ALOS PALSAR Unwrap Phase', fontsize=12, fontweight='bold')
plt.subplot(1, 2, 2)
# mask from dem
is_island = np.where(dem == 0, 0, 1)
unw_island = unw * is_island
plt.imshow(unw_island, cmap='bwr', vmin=-np.pi*num_plot_cycle, vmax=np.pi*num_plot_cycle)
plt.colorbar(shrink=0.5)
plt.title('ALOS PALSAR Unwrap Phase DEM Masked', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'unw.png'))
plt.show();plt.clf();plt.close()

In [ ]:
plt.figure(figsize=(56, 12), dpi=160)
plt.subplot(151)
plt.title('Unwrapped phase', fontsize=14, fontweight='bold')
plt.imshow(unw, cmap="plasma")
plt.colorbar(shrink=0.5)
plt.subplot(152)
plt.title('Original phase', fontsize=14, fontweight='bold')
plt.imshow(np.angle(slc_rm_topo_filter), cmap="hsv", vmin=-np.pi, vmax=np.pi)
plt.colorbar(shrink=0.5)
plt.subplot(153)
plt.title('Coherence', fontsize=14, fontweight='bold')
plt.imshow(coherence, cmap="gray", vmin=th_coherence, vmax=1)
plt.colorbar(shrink=0.5)
plt.subplot(154)
plt.title('Amplitude Difference', fontsize=14, fontweight='bold')
plt.imshow(amp_ms_crop)
plt.colorbar(shrink=0.5)
plt.subplot(155)
plt.title('Connected components', fontsize=14, fontweight='bold')
plt.imshow(conncomp, cmap="Set1")
plt.colorbar(shrink=0.5)
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'unwrapped_phase_info.png'))
plt.show();plt.clf();plt.close()


## Save Logs


In [64]:
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'unwrap.tif'), unw)
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'unwrap_island.tif'), unw_island)
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'connected_component.tif'), conncomp)
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'coherence.tif'), coherence)
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'amplitude_difference_3ch.tif'), amp_ms_crop)

# interferogram phase
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'ifgm_remove_orbital_stripe.tif'), ifgm_rm_orb)
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'ifgm_topography.tif'), ifgm_topo)
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'ifgm_topography_filtered.tif'), ifgm_topo_filtered)
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'topography.tif'), topography)

## DInSAR

In [ ]:
difference_slant = np.divide(unw_island * wave_length, (np.square(2) * np.pi))

tifffile.imwrite(os.path.join(PATH_OUTPUT, 'difference_slant.tif'), difference_slant)

spans = [0.1, 0.2, 0.3, 0.4]

plt.figure(figsize=(18, 8), dpi=120, facecolor='w', edgecolor='k')
plt.subplot(1, 3, 1)
plt.imshow(difference_slant, cmap='coolwarm', vmin=spans[0], vmax=spans[1])
plt.colorbar(shrink=0.5)
plt.title(f'ALOS PALSAR Difference Slant {spans[0]} ~ {spans[1]} [m]', fontsize=12, fontweight='bold')

plt.subplot(1, 3, 2)
plt.imshow(difference_slant, cmap='coolwarm', vmin=spans[1], vmax=spans[2])
plt.colorbar(shrink=0.5)
plt.title(f'ALOS PALSAR Difference Slant {spans[1]} ~ {spans[2]} [m]', fontsize=12, fontweight='bold')

plt.subplot(1, 3, 3)
plt.imshow(difference_slant, cmap='coolwarm', vmin=spans[2], vmax=spans[3])
plt.colorbar(shrink=0.5)
plt.title(f'ALOS PALSAR Difference Slant {spans[2]} ~ {spans[3]} [m]', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'difference_slant.png'))
plt.show();
plt.clf();plt.close()

In [66]:
from utils.interferogram import get_angle_vectors

def get_incident_angle(xyz_elev_loc, orb, idx_line):
    
    orb_idx = orb[idx_line, :]
    sla = orb_idx - xyz_elev_loc
    incident_angle = get_angle_vectors(sla, xyz_elev_loc)
    return incident_angle

incident_angle = get_incident_angle(xyz_elev_m_loc, orb=orb_m, idx_line=idx_line_elev_crop)

In [ ]:
plt.figure(figsize=(10, 6), dpi=70, facecolor='w', edgecolor='k')
plt.imshow(np.degrees(incident_angle), cmap='YlGnBu',
        #    vmin=20, vmax=70
           )
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Incident Angle [°]', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'incident_angle.png'))
plt.show();
plt.clf();plt.close()

In [ ]:
options = gdal.DEMProcessingOptions(
    slopeFormat='degree',
    )
gdal.DEMProcessing(
    os.path.join(PATH_OUTPUT, 'dem_slide_slope.tif'), 
    os.path.join(PATH_OUTPUT, 'dem_slide.tif'), 
    'slope', options=options)

slope = tifffile.imread(os.path.join(PATH_OUTPUT, 'dem_slide_slope.tif'))

plt.figure(figsize=(12, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(slope, cmap='Oranges', vmin=5, vmax=85)
plt.colorbar(shrink=0.8)
plt.title('SRTM DEM 30m Slide Slope [°]', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'slope.png'))
plt.show();
plt.clf();plt.close()


In [ ]:
slope_range_tan_minus = (np.roll(dem_slide, 1, axis=1) - dem_slide + 0.0001) * (30 / W_SCALE)
slope_range_tan_pulas = (dem_slide - np.roll(dem_slide, 1, axis=1) + 0.0001) * (30 / W_SCALE)
slope_range = np.degrees(np.arctan(slope_range_tan_pulas))/2 - np.degrees(np.arctan(slope_range_tan_minus))/2
# slope_range[slope_range == 0] = np.nan

plt.figure(figsize=(12, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(slope_range, cmap='Spectral',)
plt.colorbar(shrink=0.8)
plt.title('SRTM DEM 30m Slide Slope Range [°]', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'slope_range.png'))
plt.show();
plt.clf();plt.close()

In [ ]:
local_incident_angle = np.degrees(incident_angle) - slope_range
# local_incident_angle[dem_slide == 0] = np.nan

plt.figure(figsize=(12, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(local_incident_angle, cmap='brg', vmin=1, vmax=89)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Local Incident Angle [°]', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'local_incident_angle.png'))
plt.show();
plt.clf();plt.close()


In [ ]:
# line-of-sight ==  difference of slant

difference_w = difference_slant * np.cos(np.radians(local_incident_angle))
difference_h = difference_slant * np.sin(np.radians(local_incident_angle))

diff_range = 0.3

plt.figure(figsize=(24, 8), dpi=120, facecolor='w', edgecolor='k')
plt.subplot(1, 2, 1)
plt.imshow(difference_w, cmap='PuOr', vmin=-diff_range, vmax=diff_range)
plt.colorbar(shrink=0.8)
plt.title(f'ALOS PALSAR Difference Width -{diff_range} ~ {diff_range} [m]', fontsize=12, fontweight='bold')

plt.subplot(1, 2, 2)
plt.imshow(difference_h, cmap='PRGn', vmin=-diff_range, vmax=diff_range)
plt.colorbar(shrink=0.8)
plt.title(f'ALOS PALSAR Difference Height -{diff_range} ~ {diff_range} [m]', fontsize=12, fontweight='bold')

plt.tight_layout(pad=0.6)
plt.savefig(os.path.join(PATH_OUTPUT, f'difference_wh_{diff_range}.png'))
plt.show();
plt.clf();plt.close()

## 3D DEM

In [72]:
PATH_DEM_CUT = '../data/sakurazima/srtm_dem_sakurazima_cut_height_area.tif'

dem = tifffile.imread(PATH_DEM_CUT).astype(np.float32)
dem_slide = tifffile.imread(os.path.join(PATH_OUTPUT, 'dem_slide.tif')).astype(np.float32)

In [ ]:

fig = plt.figure(figsize=(28, 10), dpi=120, facecolor='w', edgecolor='k')
ax = fig.add_subplot(121, projection='3d')
X = np.arange(0, dem.shape[1], 1)
Y = np.arange(0, dem.shape[0], 1)
X, Y = np.meshgrid(X, Y)
ax.plot_surface(X, Y, dem, cmap='rainbow')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_zlabel('Elevation [m]')
ax.view_init(elev=20, azim=-80, roll=0)
ax.set_title('DEM [m]', fontsize=16, fontweight='bold')
ax = fig.add_subplot(122, projection='3d')
X = np.arange(0, dem_slide.shape[1], 1)
Y = np.arange(0, dem_slide.shape[0], 1)
X, Y = np.meshgrid(X, Y)
ax.plot_surface(X, Y, dem_slide, cmap='rainbow', vmax=1117)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_zlabel('Elevation [m]')
ax.view_init(elev=20, azim=-80, roll=0)
ax.set_title('DEM Slide [m]', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'dem_3d.png'))
plt.show();plt.clf();plt.close()

In [ ]:
slc_rm_orb = amp_m_crop * np.exp(1j * ifgm_rm_orb)

unw_topo = snaphu.unwrap(slc_rm_orb, coherence,
                              nlooks=32.0, cost="smooth", init="mcf", 
                              nproc=8)

In [ ]:
plt.figure(figsize=(24, 8), dpi=120, facecolor='w', edgecolor='k')
plt.subplot(1, 3, 1)
plt.imshow(-unw_topo[0], cmap='terrain',)
plt.colorbar(shrink=0.5)
plt.title('ALOS PALSAR Unwrap Phase Topography', fontsize=12, fontweight='bold')

plt.subplot(1, 3, 2)
plt.imshow(np.angle(slc_rm_orb), cmap='hsv', vmin=-np.pi, vmax=np.pi)
plt.colorbar(shrink=0.5)
plt.title('ALOS PALSAR Original Phase Topography', fontsize=12, fontweight='bold')

plt.subplot(1, 3, 3)
plt.imshow(unw_topo[1], cmap='Set2')
plt.colorbar(shrink=0.5)
plt.title('ALOS PALSAR Connected Components Topography', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'unw_topo.png'))
plt.show();
plt.clf();plt.close()

In [ ]:
# 3D plot
fig = plt.figure(figsize=(24, 16), dpi=120, facecolor='w', edgecolor='k')
ax = fig.add_subplot(121, projection='3d')

unw_topo_surface =  np.divide(-unw_topo[0] * wave_length, (np.square(2) * np.pi))

X = np.arange(0, unw_topo_surface.shape[1], 1)
Y = np.arange(0, unw_topo_surface.shape[0], 1)

X, Y = np.meshgrid(X, Y)

ax.plot_surface(X, Y, unw_topo_surface, cmap='twilight', rstride=2, cstride=2)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_zlabel('Number of Cycle [n]')
ax.view_init(elev=40, azim=-80, roll=0)

ax.set_title('Unwrap Phase Topography', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'unw_topo_3d.png'))
plt.show();plt.clf();plt.close()

## Correlation Zero

In [ ]:
delta_phi = -unw_topo[0] + 8 * np.pi

PATH_DEM_CUT = '../data/sakurazima/srtm_dem_sakurazima_cut_height_area.tif'
dem = tifffile.imread(PATH_DEM_CUT).astype(np.float32)
dem = np.where(dem < 0, 0, dem) # preprocess
alos_m.crop_imgs = amp_ms_crop
dem = alos_m.resize_crop_area(dem)

delta_phi[dem == 0] = 0

plt.figure(figsize=(10, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(delta_phi, cmap='gist_earth', vmin=0,)
plt.colorbar(shrink=0.5)
plt.title('Topography Correlation', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'unw_topo_delta.png'))
plt.show();plt.clf();plt.close()

## 桜島
標高 `1,117` [m]

In [99]:
PATCH_START_A = 4800
PATCH_END_A =   4800+delta_phi.shape[0]

B = np.linalg.norm(np.abs(orb_m - orb_s), axis=1)
B_cut = np.stack([B[PATCH_START_A:PATCH_END_A]]*delta_phi.shape[1], axis=1)

In [ ]:
h = wave_length * slant_m_cut * np.sin(incident_angle) / (4 * np.pi * np.cos(incident_angle - 0)* B_cut) * delta_phi

plt.figure(figsize=(10, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(h, cmap='terrain',
           vmin=0, vmax=1200)
plt.colorbar(shrink=0.5)
plt.title('Terrain [m]', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'unw_terrain.png'))
plt.show();plt.clf();plt.close()

In [ ]:
# filter processing
K_SIZE = 9
h_blur = cv2.GaussianBlur(h, (K_SIZE, K_SIZE), 0)

plt.figure(figsize=(10, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(h_blur, cmap='terrain',
           vmin=0, vmax=1200)
plt.colorbar(shrink=0.5)
plt.title('Terrain Blur [m]', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'unw_terrain_blur.png'))
plt.show();plt.clf();plt.close()

# median filter
h_median = scipy.ndimage.median_filter(h, size=K_SIZE)

plt.figure(figsize=(10, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(h_median, cmap='terrain',
           vmin=0, vmax=1200)
plt.colorbar(shrink=0.5)
plt.title('Terrain Median [m]', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'unw_terrain_median.png'))
plt.show();plt.clf();plt.close()

In [108]:
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'unwrap_topo.tif'), unw_topo)
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'unwrap_terrain.tif'), h)
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'unwrap_terrain_blur.tif'), h_blur)
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'unwrap_terrain_median.tif'), h_median)

In [ ]:
from osgeo import gdal, osr

def georeference(img, 
                 LT_LON, LT_LAT, RT_LON, RT_LAT, RB_LON, RB_LAT, LB_LON, LB_LAT, 
                 PATH_GEOTIF, gdal_type=gdal.GDT_Float32, ch_name='Band 1',
                 EPSG=4326, CH=1):
    (H, W) = img.shape[:2]

    driver = gdal.GetDriverByName('GTiff')
    # https://naturalatlas.github.io/node-gdal/classes/Constants%20(GDT).html
    out_dataset = driver.Create(PATH_GEOTIF, W, H, CH, gdal_type)
    
    out_band = out_dataset.GetRasterBand(CH)
    out_band.WriteArray(img)
    out_band.SetDescription(ch_name)
    
    gcp_list = [
        gdal.GCP(LT_LON, LT_LAT, 0, 0, 0),
        gdal.GCP(RT_LON, RT_LAT, 0, W, 0),
        gdal.GCP(RB_LON, RB_LAT, 0, W, H),
        gdal.GCP(LB_LON, LB_LAT, 0, 0, H),
        ]
    
    srs = osr.SpatialReference()
    srs.ImportFromEPSG(EPSG)
    out_dataset.SetGCPs(gcp_list, srs.ExportToWkt())
    out_dataset = None
    
PATH_DEM_INSAR = os.path.join(PATH_OUTPUT, 'unwrap_terrain_geo.tif')
LT_LON, LT_LAT, RT_LON, RT_LAT, RB_LON, RB_LAT, LB_LON, LB_LAT = bbox[0], bbox[1], bbox[2], bbox[1], bbox[2], bbox[3], bbox[0], bbox[3]
print(LT_LON, LT_LAT, RT_LON, RT_LAT, RB_LON, RB_LAT, LB_LON, LB_LAT)

georeference(h, 
                 LT_LON, LT_LAT, RT_LON, RT_LAT, RB_LON, RB_LAT, LB_LON, LB_LAT, 
                 PATH_DEM_INSAR, gdal_type=gdal.GDT_Float32, ch_name='InSAR Original Terrain',
                 EPSG=4326, CH=1)
PATH_DEM_MED = os.path.join(PATH_OUTPUT, 'unwrap_terrain-med_geo.tif')
georeference(h_median, 
                 LT_LON, LT_LAT, RT_LON, RT_LAT, RB_LON, RB_LAT, LB_LON, LB_LAT, 
                 PATH_DEM_MED, gdal_type=gdal.GDT_Float32, ch_name='InSAR Original Terrain Median Filter',
                 EPSG=4326, CH=1)

## おまけ

In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# TIFFファイルの読み込み
PATH_DEM_CUT = '../data/sakurazima/srtm_dem_sakurazima_cut_height_area.tif'
with rasterio.open(PATH_DEM_CUT) as src:
    dem_data = src.read(1)  # 1つ目のバンドを読み込む
    transform = src.transform  # 座標変換情報

print(dem_data.shape)
h, w = dem_data.shape

# X, Y 座標を作成
x = np.linspace(0, w, w)
y = np.linspace(0, h, h)
X, Y = np.meshgrid(x, y)

# 3Dプロット
fig = plt.figure(figsize=(20, 8), dpi=120, facecolor='w', edgecolor='k')
ax = fig.add_subplot(121, projection="3d")

# ワイヤーフレームメッシュを描画
ax.plot_wireframe(X, Y, dem_data, color='black', linewidth=0.25,)

# ラベル設定
ax.set_xlabel("Latitude")
ax.set_ylabel("Logitude")
ax.set_zlabel("Elevation")
ax.set_zlim(0, 1200)  # Z軸の範囲を 0～1200 に制限

ax = fig.add_subplot(122, projection="3d")
# flatt earth
h_zero = np.zeros_like(dem_data)
ax.plot_wireframe(X, Y, h_zero, color='black', linewidth=0.25)
ax.set_xlabel("Latitude")
ax.set_ylabel("Logitude")
ax.set_zlabel("Elevation")
ax.set_zlim(0, 1200)  # Z軸の範囲を 0～1200 に制限

plt.tight_layout()

plt.savefig(os.path.join(PATH_OUTPUT, 'dem-flat_3d_mesh.png'))
plt.show();plt.clf();plt.close()